In [8]:
import numpy as np
from numpy.linalg import eig

In [9]:
def hmm(transitions, emissions):
    hidden_states = list(transitions.keys())
    observed_states = list(next(iter(emissions.values())).keys())
    
    t_matrix = np.array([list(state.values()) for state in transitions.values()])
    e_matrix = np.array([list(state.values()) for state in emissions.values()])

    eigenvalues, eigenvectors = np.linalg.eig(t_matrix.T)
    idx = np.argmin(np.abs(eigenvalues - 1))
    v = eigenvectors[:, idx].real
    s_dist = v / v.sum()

    return {
        "A": t_matrix,
        "B": e_matrix,
        "pi": s_dist,
        "states": hidden_states,
        "observations": observed_states
    }

In [10]:
def forward_algorithm(model, observed_seq):
    A = model["A"]
    B = model["B"]
    pi = model["pi"]
    states = model["states"]
    observations = model["observations"]

    n_states = len(states)
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]

    #already computed for first time step: when i = 0
    alphas_old = [ pi[j] * B[j][obs_indices[0]] for j in range(n_states)]

    #computing for other time steps
    for i in range(1, len(observed_seq)):
        alphas_new = [0.0]*n_states
        obs_idx = obs_indices[i]

        #calculating alpha_time_i_(state j) =
        #sum of (alpha_time_i-1 for all states * probability of going from that state to state j )  * emission probability of observance with state j
        for j in range(n_states):
            alphas_new[j] = sum(alphas_old[k]* A[k][j] for k in range(n_states)) * B[j][obs_idx]
        alphas_old = alphas_new

    print(sum(alphas_old))

In [11]:
def viterbi_algorithm(model, observed_seq):
    A = model["A"]
    B = model["B"]
    pi = model["pi"]
    states = model["states"]
    observations = model["observations"]

    num_states = len(states)
    length = len(observed_seq)
    obs_to_idx = {o: i for i, o in enumerate(observations)}
    obs_indices = [obs_to_idx[o] for o in observed_seq]

    #storing alpha_time(X) and most likely state from which we came to state X
    probs = [  [[0.0] for _ in range(num_states)]  for _ in range(length)]
    backtr = [ [[0] for _ in range(num_states)] for _ in range(length)]

    #Initialisation
    for x in range(num_states):
        probs[0][x] = pi[x] * B[x][obs_indices[0]]
        backtr[0][x] = None

    #Dynamic Programming
    for time in range(1, length):
        observed_idx = obs_indices[time]

        #Finding the probability, and previous_state of the most probable path to (time_X)
        for x in range(num_states):
            max_p = -1.0
            arg_max = None

            for prev_state in range(num_states):
                term = probs[time-1][prev_state] * A[prev_state][x]
                if term > max_p:
                    max_p = term
                    arg_max = prev_state
            
            #storing the Alpha_time(X) probability, and most likely previous state from which it came
            probs[time][x] = max_p * B[x][observed_idx] #multiplying everything through by P(observed | state) later
            backtr[time][x] = arg_max
    
    #Backtracking
    idx_seq = [0]*length

    #Find most likely hidden state on last day
    last = max(range(num_states), key=lambda x: probs[length-1][x])
    idx_seq[length-1] = last
    #working backwards to find most likely sequence
    for time in range(length-2, -1, -1):
        idx_seq[time] = backtr[time+1][last]
        last = backtr[time+1][last]
    
    #producing sequence, from idx_seq
    seq = [""]*length
    for time, idx in enumerate(idx_seq):
        seq[time] = states[idx]
    print(seq)


In [12]:
transitions ={
    "sunny": {"sunny": 0.6, "rainy": 0.3, "cloudy": 0.1},
    "rainy": {"sunny": 0.1, "rainy": 0.5, "cloudy": 0.4},
    "cloudy": {"sunny": 0.3, "rainy": 0.4, "cloudy": 0.3},
}
emissions = {
    "sunny": {"happy": 0.8, "sad": 0.1, "neutral": 0.1},
    "rainy": {"happy": 0.2, "sad": 0.6, "neutral": 0.2},
    "cloudy": {"happy": 0.4, "sad": 0.4, "neutral": 0.2},
}
model_1 = hmm(transitions, emissions)


observed_seq = ["happy", "sad", "neutral"]

#forward_algorithm(model_1, observed_seq) #forward algorithm tells you how likely the provided observed sequence is
forward_algorithm(model_1, observed_seq)

#viterbi tells you the most likely state sequence which corresponds to the provided observed sequence
viterbi_algorithm(model_1, observed_seq) 

0.02715147540983607
['sunny', 'rainy', 'rainy']
